In [1]:
## This notebook investigates the phenotypes for various properties

In [2]:
# Requirements

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
from sklearn.cluster import KMeans
from scipy.spatial.distance import pdist, squareform
import xgboost as xgb
from sklearn.svm import SVR
from sklearn.inspection import permutation_importance
import shap
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from scipy.stats import shapiro
from scipy.stats import pearsonr
import gc
from sklearn.model_selection import KFold

/Users/madisoncreach/miniconda3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# read in the dataframe and set the GeneID as the index

path = ('/Users/madisoncreach/Prediction_Project/Input_Datasets/full_phenos.csv')
phenos = pd.read_csv(path, header=0)
full_phenos = phenos.set_index('GeneID')

In [5]:
# Store traits that fail the normality test
non_normal_traits = []

# Loop through each column (trait)
for col in full_phenos.columns:
    # Drop NaNs before testing
    data = full_phenos[col].dropna()

    # Perform Shapiro-Wilk test
    stat, p = shapiro(data)

    # If p < 0.05, reject normality
    if p < 0.05:
        non_normal_traits.append(col)

/var/folders/zb/nzrt7t694fggk1r6y6m3486w0000gn/T/ipykernel_18889/382203992.py:10: UserWarning: scipy.stats.shapiro: Input data has range zero. The results may not be accurate.
  stat, p = shapiro(data)


In [6]:
print(non_normal_traits)

['AnthesisGDD_L', 'ASIGDD_L', 'ASI_L', 'SAMheight_M', 'SAMRadius_M', 'SAMVolume_M', 'BraceRoot_N', 'StalkArea_N', 'diffStalkDiamThickThin_N', 'Lignin_N', 'SAMVolume1_N', 'SAMArea_N', 'SAMParabolicCoefficient_N', 'SAMSurfaceArea_N', 'SAMarcLength_N', 'RootAngle1_O', 'RootAngle2_O', 'TasselOpenness_P', 'TasselLengthGT_P', 'BranchZoneLengthGT_P', 'LowestBranchAngleGT_P', 'BranchNumber_P', 'BranchZoneLengthAuto_P', 'LowestBranchAngleAuto_P', 'TasselLengthError_P', 'CentralSpikeLengthError_P', 'BranchZoneLengthError_P', 'LowestBranchLengthError_P', 'LowestBranchAngleError_P', 'RootAngle_I', 'Anthesis_J', 'Silking_J', 'RootLodgingPct_J', 'StalkLodgingPct_J', 'LeafAngle_J', 'EarsPerPlant_J', 'CobWeightGrams_J', 'HundredKernelMassGrams_J', 'TotalGrainMassGrams_J', 'BushelAcreEquivalent_J', 'GrainPercentMoisture_J', 'EarWidth_J', 'EarFilledLength_J', 'KernelRowNumber_J', 'KernelsPerRow_J', 'PercentFill_J', 'SouthernRustSeverityScore_J', 'PlantHeight_J', 'ExtantLeafNumber1_J', 'NodesWithBraceRoo

In [17]:
skew_values = full_phenos.skew()

# Define a threshold for "high skew" (e.g., > |1| is commonly considered high)
high_skew = skew_values[skew_values.abs() > 1]

# Print traits with high skew
print("Highly skewed traits:")
print(high_skew)

Highly skewed traits:
SAMVolume_M                    1.538035
Lignin_N                       1.012626
SAMVolume1_N                   1.435200
SAMArea_N                      1.032184
SAMSurfaceArea_N               1.013318
BranchZoneLengthGT_P           1.332823
LowestBranchAngleGT_P          1.831730
LowestBranchAngleAuto_P        1.138081
TasselLengthError_P           -2.268320
BranchZoneLengthError_P       -1.679225
LowestBranchAngleError_P      -1.099828
RootLodgingPct_J               1.021130
StalkLodgingPct_J              1.932301
EarsPerPlant_J                -1.589441
PercentFill_J                 -2.522857
NodesWithBraceRoots_J          1.058566
TillersPerPlant_J              7.932535
BranchesPerTassel_J            1.092671
TasselLength_J                 2.235418
TasselSpikeLength_J            2.159275
SMVAUPDC_B                    -1.798131
SMV10DAI_B                    -1.291893
SMV14DAI_B                    -1.790870
SMV21DAI_B                    -2.225038
SMV28DAI_B        

In [18]:
stds = full_phenos.std()
low_variance_traits = stds[stds < 0.1] 

In [19]:
print(low_variance_traits)

EHdivPH_L                      0.060379
diffStalkDiamThickThin_N       0.048151
Lignin_N                       0.033883
SAMParabolicCoefficient_N      0.000627
EarsPerPlant_J                 0.044469
TillersPerPlant_J              0.002047
SpikeProportion_C              0.070309
FractalDimension_C             0.038900
Toruosity_C                    0.011144
RindThickness_D                0.054631
VascularBundleArea_D           0.000000
senescence2_F                  0.033895
senescence3_F                  0.041478
LeafCuticularConductance4_H    0.027422
LeafCuticularConductance5_H    0.012656
LeafCuticularConductance6_H    0.031768
LeafCuticularConductance7_H    0.028759
Ash_K                          0.095125
Fructose_K                     0.011042
Glucose_K                      0.031547
Sucrose_K                      0.030160
dtype: float64
